## Polytope Climate-DT AOI example notebook

This notebook shows how to use earthkit-data and earthkit-plots to pull destination-earth data from LUMI and plot it using earthkit-plots.

It also shows how to cut-out an area of interest using earthkit-transforms and perform some calculations upon that data.

Before running the notebook you need to set up your credentials. See the main readme of this repository for different ways to do this or use the following cells to authenticate.

You will need to generate your credentials using the desp-authentication.py script.

This can be run as follows:

In [ ]:
%%capture cap
%run ../desp-authentication.py

This will generate a token that can then be used by earthkit and polytope.

In [ ]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]

# Requirements
To run this notebook install the following:
* pip install earthkit-data
* pip install earthkit-plots
* pip install earthkit-transforms
* pip install earthkit-regrid
* pip install cf-units         (Optional for unit conversion in plots)
* pip install matplotlib.pyplot

If you do not have eccodes installed please install eccodes using conda as it is a dependency of earthkit, or install earthkit via conda

* conda install eccodes -c conda-forge
* conda install earthkit-data -c conda-forge

In [ ]:
import earthkit.data
import earthkit.plots
import earthkit.regrid
from earthkit.transforms import aggregate as ek_aggregate

from earthkit.data.testing import earthkit_remote_test_data_file
import earthkit.plots.quickmap as qmap
import matplotlib.pyplot as plt
from polytope.api import Client

In [ ]:
# Defaults to making a live data request. Set to false to use the cached GRIB file instead.
import os

LIVE_REQUEST = os.getenv("LIVE_REQUEST", "true").lower() == "true"
LIVE_REQUEST

In [ ]:
request = {
    'activity': 'ScenarioMIP',
    'class': 'd1',
    'dataset': 'climate-dt',
    'date': '20200102',
    'experiment': 'SSP3-7.0',
    'expver': '0001',
    'generation': '1',
    'levtype': 'sfc',
    'model': 'IFS-NEMO',
    'param': '134/165/166/167',
    'realization': '1',
    'resolution': 'standard',
    'stream': 'clte',
    'time': '0100', # '0100/0200/0300/0400/0500/0600'
    'type': 'fc'
}

In [ ]:
data_file = "data/climate-dt-earthkit-aoi-example.grib"
if LIVE_REQUEST:
    data = earthkit.data.from_source("polytope", "destination-earth", request, address="polytope.lumi.apps.dte.destination-earth.eu", stream=False)
    data.to_target("file", data_file)
else:
    data = earthkit.data.from_source("file", data_file)

In [ ]:
data.ls()

In [ ]:
chart = earthkit.plots.Map(domain="Europe")
chart.grid_cells(data[0])
chart.legend()
chart.gridlines()
chart.title()
chart.coastlines()
chart.show()


In [ ]:
# Regrid t=from healpix for conversion to xarray
data_latlon = earthkit.regrid.interpolate(data, out_grid={"grid": [0.1,0.1]}, method="linear")

In [ ]:
# Convert data to xarray
xarr = data_latlon[3].to_xarray()
xarr

In [ ]:
chart = earthkit.plots.Map(domain="Europe")
chart.grid_cells(data_latlon[0])
chart.legend()
chart.gridlines()
chart.title()
chart.coastlines()
chart.show()

Here we pull shape files for masking of our area of interest

In [ ]:
# remote_nuts_url = earthkit_remote_test_data_file("test-data", "NUTS_RG_60M_2021_4326_LEVL_0.geojson")
# nuts_data = earthkit.data.from_source("url", remote_nuts_url)
# nuts_data.save("data/NUTS_RG_60M_2021_4326_LEVL_0.geojson")

nuts_data = earthkit.data.from_source("file", "data/NUTS_RG_60M_2021_4326_LEVL_0.geojson")

In [ ]:
nuts_data.to_pandas()[:5]

`spatial.mask` applies all the features in the geometry object (`nuts_data`) to the data object (`xarr`). It returns an xarray object the same shape and type as the input xarray object with all points outside of the geometry masked.

In [ ]:
single_masked_data = ek_aggregate.spatial.mask(xarr, nuts_data)
single_masked_data


We can now plot the original data against our masked data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,4))
xarr["2t"].plot(ax=axes[0])
axes[0].set_title('Original data')
axes[0].set_xlim([0, 30])  # Longitude range for Europe
axes[0].set_ylim([35, 70])   # Latitude range for Europe

# Single masked data
single_masked_data["2t"].mean(dim='index').plot(ax=axes[1])
axes[1].set_title('Masked data')
axes[1].set_xlim([0, 30])  # Longitude range for Europe
axes[1].set_ylim([35, 70])   # Latitude range for Europe


We can further mask to only get the data for a given country.

In [ ]:
masked_data = ek_aggregate.spatial.mask(xarr, nuts_data, mask_dim="FID")
masked_data

Here we only retrieve data for Germany, and plot.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,3))
xarr["2t"].plot(ax=axes[0])
axes[0].set_title('Original data')
axes[0].set_xlim([0, 30])  # Longitude range for Europe
axes[0].set_ylim([35, 70])   # Latitude range for Europe
masked_data["2t"].sel(FID='DE').plot(ax=axes[1])
axes[1].set_title('Masked for Germany')
axes[1].set_xlim([0, 30])  # Longitude range for Europe
axes[1].set_ylim([35, 70])   # Latitude range for Europe
germany_data = masked_data.sel(FID='DE').dropna(dim='latitude', how='all').dropna(dim='longitude', how='all')
germany_data["2t"].plot(ax=axes[2])
axes[2].set_title('Masked Germany Zoom')